# Python Backend & REST API Deep Dive (Copilot Recreated)

This notebook recreates the richer version we built earlier in this chat: concept-first explanations plus backend-engineering patterns and runnable examples.

## What you’ll learn
- REST fundamentals and HTTP method semantics
- Authentication vs authorization in real APIs
- Practical usage with Python `requests`
- Backend engineering concerns: retries, pagination, rate-limits, and error mapping
- Production patterns: tracing, service layer, and mock-based tests

## Setup & Imports

We’ll use:
- `requests` for API calls
- `os` for environment variables
- `typing` for clearer helper signatures

Install once if needed:

```bash
pip install requests
```

> Always set timeouts and avoid hardcoding tokens in notebooks.

## Module 1 — REST Fundamentals

A REST API models **resources** and uses HTTP methods consistently.

### Core request/response model
- Request = method + URL + headers + optional body
- Response = status code + headers + body

### Method semantics
- `GET`: read-only, safe
- `POST`: create/action, not idempotent
- `PUT`: full replacement, usually idempotent
- `PATCH`: partial update
- `DELETE`: remove, usually idempotent

### Status-code intuition
- `2xx` success
- `4xx` client-side issue
- `5xx` server-side issue

In [1]:
import os
import time
import random
import uuid
from dataclasses import dataclass
from typing import Any, Dict, Iterable, List, Optional

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

print(f"requests version: {requests.__version__}")


def debug_response(resp: requests.Response, preview: int = 160) -> None:
    print("=" * 60)
    print("URL:", resp.url)
    print("Status:", resp.status_code)
    print("Content-Type:", resp.headers.get("Content-Type"))
    print("Body preview:", resp.text.replace("\n", " ")[:preview], "...")


root_resp = requests.get("https://api.github.com", timeout=20)
debug_response(root_resp)

user_resp = requests.get("https://api.github.com/users/octocat", timeout=20)
debug_response(user_resp)
if user_resp.ok:
    u = user_resp.json()
    print("login:", u.get("login"), "public_repos:", u.get("public_repos"))

requests version: 2.32.5
URL: https://api.github.com/
Status: 200
Content-Type: application/json; charset=utf-8
Body preview: {"current_user_url":"https://api.github.com/user","current_user_authorizations_html_url":"https://github.com/settings/connections/applications{/client_id}","aut ...
URL: https://api.github.com/users/octocat
Status: 200
Content-Type: application/json; charset=utf-8
Body preview: {"login":"octocat","id":583231,"node_id":"MDQ6VXNlcjU4MzIzMQ==","avatar_url":"https://avatars.githubusercontent.com/u/583231?v=4","gravatar_id":"","url":"https: ...
login: octocat public_repos: 8


## Module 2 — Authentication & Authorization

- **Authentication**: who are you?
- **Authorization**: what are you allowed to do?

Common auth patterns:
- API key
- Bearer token
- Basic auth (limited use)
- OAuth2/JWT

For GitHub examples here, use `GITHUB_TOKEN` in environment if you want authenticated calls.

In [2]:
BASE_GITHUB_API = "https://api.github.com"


def create_github_session() -> requests.Session:
    token = os.getenv("GITHUB_TOKEN")
    s = requests.Session()
    s.headers.update(
        {
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
        }
    )
    if token:
        s.headers["Authorization"] = f"Bearer {token}"
    return s


def request_json(
    session: requests.Session,
    method: str,
    url: str,
    *,
    params: Optional[Dict[str, Any]] = None,
    json_body: Optional[Dict[str, Any]] = None,
) -> Optional[Dict[str, Any]]:
    try:
        resp = session.request(method=method, url=url, params=params, json=json_body, timeout=20)
        print(f"{method} {url} -> {resp.status_code}")
        resp.raise_for_status()
        if not resp.text.strip():
            return None
        return resp.json()
    except requests.HTTPError as e:
        print("HTTP error:", e)
    except requests.RequestException as e:
        print("Request error:", e)
    except ValueError:
        print("Response not valid JSON")
    return None


sess = create_github_session()
repo = request_json(sess, "GET", f"{BASE_GITHUB_API}/repos/octocat/Hello-World")
if repo:
    print("Repo:", repo.get("full_name"), "default branch:", repo.get("default_branch"))

me = request_json(sess, "GET", f"{BASE_GITHUB_API}/user")
if me:
    print("Authenticated as:", me.get("login"))
else:
    print("Tip: set GITHUB_TOKEN to test authenticated endpoint.")

_ = request_json(sess, "POST", "https://httpbin.org/post", json_body={"topic": "REST", "level": "backend"})
_ = request_json(sess, "PATCH", "https://httpbin.org/patch", json_body={"progress": "good"})
_ = request_json(sess, "DELETE", "https://httpbin.org/delete")

GET https://api.github.com/repos/octocat/Hello-World -> 200
Repo: octocat/Hello-World default branch: master
GET https://api.github.com/user -> 401
HTTP error: 401 Client Error: Unauthorized for url: https://api.github.com/user
Tip: set GITHUB_TOKEN to test authenticated endpoint.
POST https://httpbin.org/post -> 200
PATCH https://httpbin.org/patch -> 200
DELETE https://httpbin.org/delete -> 200


## Module 3 — Backend Engineering Patterns

A backend-ready API client should include:
- explicit timeout strategy
- selective retries (transient failures only)
- error mapping to domain exceptions
- pagination handling
- observability (logs, request IDs)

### Reliability checklist
- Retry on `429`, `5xx`, and connection/read timeouts
- Keep token logic centralized
- Parse rate-limit headers and react accordingly
- Use service layer wrappers around transport client

In [3]:
class ApiClientError(Exception):
    pass


class UnauthorizedError(ApiClientError):
    pass


class ForbiddenError(ApiClientError):
    pass


class NotFoundError(ApiClientError):
    pass


class RateLimitedError(ApiClientError):
    pass


class UpstreamServerError(ApiClientError):
    pass


@dataclass
class RateLimitInfo:
    limit: int | None
    remaining: int | None
    reset_epoch: int | None


class GitHubClient:
    def __init__(self, token: str | None = None, timeout: int = 20):
        self.base_url = "https://api.github.com"
        self.timeout = timeout
        self.session = requests.Session()
        self.session.headers.update(
            {
                "Accept": "application/vnd.github+json",
                "X-GitHub-Api-Version": "2022-11-28",
                "User-Agent": "copilot-content-client/1.0",
            }
        )
        if token:
            self.session.headers["Authorization"] = f"Bearer {token}"

        retry = Retry(
            total=3,
            connect=3,
            read=3,
            backoff_factor=0.5,
            status_forcelist=[429, 500, 502, 503, 504],
            allowed_methods=["GET", "HEAD", "OPTIONS", "PUT", "DELETE", "PATCH"],
            raise_on_status=False,
        )
        adapter = HTTPAdapter(max_retries=retry)
        self.session.mount("https://", adapter)
        self.session.mount("http://", adapter)

    def _handle_errors(self, resp: requests.Response) -> None:
        if resp.status_code == 401:
            raise UnauthorizedError("Invalid or missing token")
        if resp.status_code == 403:
            raise ForbiddenError("Forbidden")
        if resp.status_code == 404:
            raise NotFoundError("Not found")
        if resp.status_code == 429:
            raise RateLimitedError("Rate limited")
        if resp.status_code >= 500:
            raise UpstreamServerError(f"Upstream error: {resp.status_code}")
        resp.raise_for_status()

    def _request(self, method: str, path: str, **kwargs) -> requests.Response:
        resp = self.session.request(method, f"{self.base_url}{path}", timeout=self.timeout, **kwargs)
        self._handle_errors(resp)
        return resp

    def rate_limit(self) -> RateLimitInfo:
        resp = self._request("GET", "/rate_limit")
        core = resp.json().get("resources", {}).get("core", {})
        return RateLimitInfo(core.get("limit"), core.get("remaining"), core.get("reset"))

    def get_repo(self, owner: str, repo: str) -> dict:
        return self._request("GET", f"/repos/{owner}/{repo}").json()

    def iter_user_repos(self, username: str, per_page: int = 30, max_pages: int = 2) -> Iterable[dict]:
        for page in range(1, max_pages + 1):
            data = self._request("GET", f"/users/{username}/repos", params={"per_page": per_page, "page": page}).json()
            if not data:
                break
            for item in data:
                yield item


client = GitHubClient(token=os.getenv("GITHUB_TOKEN"), timeout=20)
rl = client.rate_limit()
print("Rate limit:", rl)
repo_data = client.get_repo("octocat", "Hello-World")
print("Repo:", repo_data.get("full_name"), "visibility:", "private" if repo_data.get("private") else "public")
print("Recent repos:")
for i, repo_item in enumerate(client.iter_user_repos("octocat", per_page=5, max_pages=1), start=1):
    print(f"{i}. {repo_item.get('name')}")

Rate limit: RateLimitInfo(limit=60, remaining=54, reset_epoch=1774772952)
Repo: octocat/Hello-World visibility: public
Recent repos:
1. boysenberry-repo-1
2. git-consortium
3. hello-worId
4. Hello-World
5. linguist


## Module 4 — Production Patterns (Tracing + Service Layer + Mock Tests)

### Why this matters
- tracing helps debug distributed failures
- service layer keeps business logic clean
- mock tests validate behavior without relying on external network stability

In [4]:
@dataclass
class RequestLog:
    request_id: str
    method: str
    path: str
    status: int
    duration_ms: int


def compute_backoff_seconds(attempt: int, base: float = 0.3, cap: float = 8.0) -> float:
    raw = min(cap, base * (2 ** (attempt - 1)))
    return raw + random.uniform(0, raw * 0.25)


def request_with_trace(client: GitHubClient, method: str, path: str, **kwargs) -> tuple[dict | None, RequestLog]:
    request_id = str(uuid.uuid4())
    headers = kwargs.pop("headers", {}) or {}
    headers["X-Request-ID"] = request_id

    start = time.perf_counter()
    resp = client.session.request(method, f"{client.base_url}{path}", headers=headers, timeout=client.timeout, **kwargs)
    duration_ms = int((time.perf_counter() - start) * 1000)
    log = RequestLog(request_id, method, path, resp.status_code, duration_ms)

    client._handle_errors(resp)

    data = None
    if resp.text.strip():
        try:
            data = resp.json()
        except ValueError:
            data = None
    return data, log


class RepoService:
    def __init__(self, gh_client: GitHubClient):
        self.gh_client = gh_client

    def get_repo_summary(self, owner: str, repo: str) -> dict:
        d = self.gh_client.get_repo(owner, repo)
        return {
            "full_name": d.get("full_name"),
            "private": d.get("private"),
            "default_branch": d.get("default_branch"),
            "stargazers_count": d.get("stargazers_count"),
            "open_issues_count": d.get("open_issues_count"),
        }


svc = RepoService(client)
summary = svc.get_repo_summary("octocat", "Hello-World")
print("Repo summary:", summary)

rate_data, req_log = request_with_trace(client, "GET", "/rate_limit")
print("Request log:", req_log)
if rate_data:
    core = rate_data.get("resources", {}).get("core", {})
    print("Rate remaining:", core.get("remaining"), "/", core.get("limit"))

print("Backoff samples:")
for a in range(1, 5):
    print(f" attempt {a}: {compute_backoff_seconds(a):.3f}s")


# lightweight mock-based checks
class _MockResp:
    def __init__(self, status_code: int):
        self.status_code = status_code
        self.text = "{}"

    def raise_for_status(self):
        if self.status_code >= 400:
            raise requests.HTTPError(f"{self.status_code} Error")


def _assert_mapping(status: int, exc_type: type[Exception]) -> str:
    try:
        client._handle_errors(_MockResp(status))
    except exc_type:
        return f"PASS: {status} -> {exc_type.__name__}"
    return f"FAIL: {status} mapping"


print(_assert_mapping(404, NotFoundError))
print(_assert_mapping(429, RateLimitedError))

Repo summary: {'full_name': 'octocat/Hello-World', 'private': False, 'default_branch': 'master', 'stargazers_count': 3549, 'open_issues_count': 3780}
Request log: RequestLog(request_id='4c71381b-9aa7-479c-86a3-731f81c3c9ac', method='GET', path='/rate_limit', status=200, duration_ms=36)
Rate remaining: 51 / 60
Backoff samples:
 attempt 1: 0.342s
 attempt 2: 0.728s
 attempt 3: 1.311s
 attempt 4: 2.564s
PASS: 404 -> NotFoundError
PASS: 429 -> RateLimitedError
